# 🌿 Plant Disease Detection - Evaluation
Evaluasi dan perbandingan Baseline vs Proposed Model

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = '/content/drive/MyDrive/plant_disease'
DATASET_PATH = '/content/plantvillage/plantvillage dataset/color'
import json, os
with open(f'{SAVE_PATH}/class_indices.json') as f:
    class_idx = json.load(f)
CLASS_NAMES = [k.split('___')[1].replace('_',' ') for k in sorted(class_idx, key=class_idx.get)]
print(f'Jumlah kelas: {len(CLASS_NAMES)}')

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_datagen = ImageDataGenerator(rescale=1./255)
val_gen = ImageDataGenerator(rescale=1./255, validation_split=0.2).flow_from_directory(
    DATASET_PATH, target_size=(224,224), batch_size=32,
    class_mode='categorical', subset='validation', seed=42, shuffle=False
)
print(f'Val samples: {val_gen.samples}')

In [ ]:
# Load models
baseline = tf.keras.models.load_model(f'{SAVE_PATH}/models/baseline_best.h5')
proposed = tf.keras.models.load_model(f'{SAVE_PATH}/models/proposed_best.h5')
print('Models loaded!')

In [ ]:
def evaluate_model(model, gen, name):
    gen.reset()
    preds = model.predict(gen, verbose=1)
    y_pred = np.argmax(preds, axis=1)
    y_true = gen.classes[:len(y_pred)]
    report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
    print(f'\n{"="*60}')
    print(f'HASIL: {name}')
    print(f'{"="*60}')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    return y_true, y_pred, report

y_true_b, y_pred_b, rep_b = evaluate_model(baseline, val_gen, 'Baseline EfficientNetB4')
val_gen.reset()
y_true_p, y_pred_p, rep_p = evaluate_model(proposed, val_gen, 'Proposed EfficientNetB4+CBAM')

In [ ]:
# ── Confusion Matrix ──
def plot_cm(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(20, 16))
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linewidths=0.5)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.xticks(rotation=90, fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()

plot_cm(y_true_b, y_pred_b, 'Confusion Matrix - Baseline', f'{SAVE_PATH}/visualisasi/cm_baseline.png')
plot_cm(y_true_p, y_pred_p, 'Confusion Matrix - Proposed (CBAM)', f'{SAVE_PATH}/visualisasi/cm_proposed.png')

In [ ]:
# ── Comparison Table & Graph ──
metrics = ['Accuracy','Precision','Recall','F1-Score']
b_scores = [rep_b['accuracy'], rep_b['weighted avg']['precision'],
            rep_b['weighted avg']['recall'], rep_b['weighted avg']['f1-score']]
p_scores = [rep_p['accuracy'], rep_p['weighted avg']['precision'],
            rep_p['weighted avg']['recall'], rep_p['weighted avg']['f1-score']]

# Save CSV
df_result = pd.DataFrame({'Metric': metrics, 'Baseline': b_scores, 'Proposed_CBAM': p_scores})
df_result['Improvement (%)'] = ((df_result['Proposed_CBAM'] - df_result['Baseline']) / df_result['Baseline'] * 100).round(2)
df_result.to_csv(f'{SAVE_PATH}/hasil_eksperimen/comparison_result.csv', index=False)
print(df_result.to_string(index=False))

# Comparison Chart
x = np.arange(len(metrics))
fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x-0.2, b_scores, 0.35, label='Baseline (EfficientNetB4)', color='#1565c0', alpha=0.85)
bars2 = ax.bar(x+0.2, p_scores, 0.35, label='Proposed (EfficientNetB4+CBAM)', color='#2e7d32', alpha=0.85)
for bar in [*bars1, *bars2]:
    ax.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.002,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylabel('Score'); ax.set_title('Perbandingan Baseline vs Proposed Method', fontsize=13, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=11)
ax.legend(fontsize=10); ax.set_ylim(0.85, 1.01); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_PATH}/visualisasi/comparison_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Semua evaluasi selesai dan tersimpan!')